# 02. Delta Lake 실습 — MERGE, Time Travel, 스키마 강제, OPTIMIZE

실습용 복사본 테이블 `orders_lab`을 만들어 바꿔 봅니다. 원본 테이블을 망가뜨렸다면 `reset_practice_tables()`로 되돌릴 수 있습니다.

In [ ]:
%%sql
CREATE OR REPLACE TABLE orders_lab AS SELECT * FROM orders;
DESCRIBE DETAIL orders_lab

## UPDATE / DELETE — 일반 Parquet 테이블에서는 불가능한 작업

In [ ]:
%%sql
UPDATE orders_lab SET status = 'refunded' WHERE order_id = 50001;
DELETE FROM orders_lab WHERE status = 'cancelled';
SELECT status, COUNT(*) AS n FROM orders_lab GROUP BY status

## MERGE INTO (Upsert)
결제사에서 받은 상태 변경 파일을 반영한다고 가정합니다: 기존 주문은 상태를 업데이트하고, 새 주문은 추가합니다.

In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW status_updates AS
SELECT * FROM VALUES
  (50002, 'refunded'),
  (50003, 'refunded'),
  (99999, 'completed')
AS t(order_id, status);

MERGE INTO orders_lab AS t
USING (
  SELECT u.order_id, u.status, o.user_id, o.order_ts, o.payment_method, o.coupon_code, o.discount_amount, o.total_amount
  FROM status_updates u LEFT JOIN orders o USING (order_id)
) AS s
ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET t.status = s.status
WHEN NOT MATCHED THEN INSERT (order_id, user_id, order_ts, status, payment_method, coupon_code, discount_amount, total_amount)
  VALUES (s.order_id, 1001, TIMESTAMP'2025-12-31 23:00:00', s.status, 'card', NULL, 0, 10000);

SELECT * FROM orders_lab WHERE order_id IN (50001, 50002, 50003, 99999)

## 변경 이력과 Time Travel

In [ ]:
%%sql
DESCRIBE HISTORY orders_lab

In [ ]:
%%sql
SELECT 'v0 (최초)' AS version, COUNT(*) AS n, SUM(CASE WHEN status = 'refunded' THEN 1 ELSE 0 END) AS refunded FROM orders_lab VERSION AS OF 0
UNION ALL
SELECT '현재', COUNT(*), SUM(CASE WHEN status = 'refunded' THEN 1 ELSE 0 END) FROM orders_lab

In [ ]:
%%sql
RESTORE TABLE orders_lab TO VERSION AS OF 0;
SELECT COUNT(*) AS n_after_restore FROM orders_lab

## 스키마 강제(Schema enforcement)와 스키마 진화(mergeSchema)
Delta는 테이블 스키마와 다른 데이터를 쓰려고 하면 막아서 데이터 오염을 방지합니다.

In [ ]:
from pyspark.sql import functions as F

extra = spark.table("orders_lab").limit(3).withColumn("channel_hint", F.lit("app"))
try:
    extra.write.format("delta").mode("append").saveAsTable("orders_lab")
except Exception as e:
    print("스키마 불일치로 거부됨 →", str(e).splitlines()[0][:160])

# 의도한 변경이라면 mergeSchema 옵션으로 컬럼 추가 허용
extra.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("orders_lab")
spark.catalog.refreshTable("orders_lab")
spark.table("orders_lab").printSchema()   # channel_hint 컬럼이 추가됨

## OPTIMIZE / ZORDER / VACUUM
- `OPTIMIZE`: 작은 파일을 합쳐 읽기 성능 향상
- `ZORDER BY`: 자주 필터하는 컬럼 기준으로 데이터를 모아 데이터 스키핑 향상 (Databricks에서는 Liquid Clustering `CLUSTER BY`를 권장)
- `VACUUM`: 보존 기간이 지난 옛 파일 삭제 → **그 이전 버전으로는 Time Travel 불가**

In [ ]:
%%sql
OPTIMIZE orders_lab ZORDER BY (user_id)

In [ ]:
# 데모를 위해 보존 기간 0시간으로 VACUUM (실무에서는 절대 이렇게 하지 마세요: 기본 7일)
spark.sql("VACUUM orders_lab RETAIN 0 HOURS")
try:
    # COUNT(*)는 Delta 로그 통계만으로 답할 수 있어 실제 파일을 읽는 쿼리로 확인
    spark.sql("SELECT status, COUNT(*) FROM orders_lab VERSION AS OF 1 GROUP BY status").show()
except Exception as e:
    msg = str(e)
    reason = "옛 버전의 데이터 파일이 삭제됨 (FileNotFound)" if "FileNotFound" in msg or "does not exist" in msg else msg.splitlines()[0][:160]
    print("VACUUM 이후 옛 버전 조회 실패 →", reason)

## 면접 포인트 정리
- Delta = Parquet + 트랜잭션 로그(`_delta_log`) → ACID, Time Travel, MERGE/UPDATE/DELETE
- 실수 복구: `DESCRIBE HISTORY` → `RESTORE TABLE ... TO VERSION AS OF n`
- `VACUUM`은 Time Travel 범위를 줄인다
- 스키마 강제로 잘못된 데이터 적재를 막고, 필요할 때만 `mergeSchema`

In [ ]:
# 실습 테이블 정리
spark.sql("DROP TABLE IF EXISTS orders_lab")